# bottleneck-latent-projection — worked example 1: Flatten conv features and project to latent through a hidden Linear

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `bottleneck-latent-projection`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

The end-of-encoder bottleneck takes a conv feature map `(B, C, H, W)`, flattens the channel/spatial dims into one vector with `Rearrange('b c h w -> b (c h w)')`, then runs two Linears with a ReLU between: `Linear(C*H*W, hidden)` then `Linear(hidden, latent)`. The hidden layer adds non-linear capacity; the final latent projection has **no** activation so the latent code stays unbounded.

## Worked solution

**Step 1 - flatten.** A conv stack outputs `(B, C, H, W)`. A `Linear` only accepts `(B, features)`, so we collapse the last three axes: `rearrange(x, 'b c h w -> b (c h w)')`. With `C=8, H=4, W=4` this gives `(B, 128)`. We use einops rather than `.view` so the intended axis order is explicit and won't silently transpose.

**Step 2 - first Linear + ReLU.** `flat @ W1.T + b1` maps `128 -> hidden`. PyTorch stores `nn.Linear` weight as `(out_features, in_features)`, so we transpose `W1` to make the matmul conform. We then apply `relu` to introduce the non-linearity that lets the bottleneck learn a curved manifold rather than a pure linear subspace.

**Step 3 - latent Linear, no activation.** `h @ W2.T + b2` maps `hidden -> latent`. Crucially there is **no** ReLU/sigmoid here: the latent must be free to take any real value, otherwise we'd clip half the latent space at zero.

**Step 4 - check.** We build random weights, run a `(B=5, 8, 4, 4)` batch through, and print the resulting latent shape `(5, latent)` to confirm the pipeline conforms.

In [ ]:
def encode_bottleneck(x, W1, b1, W2, b2):
    flat = rearrange(x, 'b c h w -> b (c h w)')
    h = t.relu(flat @ W1.T + b1)
    z = h @ W2.T + b2
    return z

t.manual_seed(0)
B, C, H, W = 5, 8, 4, 4
in_features = C * H * W
hidden, latent = 32, 4
x = t.randn(B, C, H, W)
W1 = t.randn(hidden, in_features)
b1 = t.randn(hidden)
W2 = t.randn(latent, hidden)
b2 = t.randn(latent)
z = encode_bottleneck(x, W1, b1, W2, b2)
print('latent shape:', tuple(z.shape))
print('first code:', z[0].round(decimals=3).tolist())